# SIMT 编程 API

## 概述

前面五节我们已经走完了 SIMT 编程模型：硬件架构、线程架构、核函数、内存层级、同步机制。本节是这一系列的收尾——把 Ascend C **SIMT 编程**场景下提供的各类 API 整理成一张「学习地图」，帮助你在真实算子开发中，知道「遇到什么问题该去查哪类 API」。

### 前置要求

- 已学习 SIMT 硬件架构、线程架构、核函数、内存层级和同步机制。
- 能够根据开发问题判断需要关注线程索引、访存、同步、原子操作或数学函数。
- 本小节为 API 学习地图，不依赖在线硬件环境。

### 学习目标

学完本小节后，你应该能够：

- 说清楚 SIMT 编程 API 的主要分类。
- 根据开发任务选择合适的 API 类别，而不是逐个死记接口。
- 理解同步与内存栅栏、原子操作、Warp 函数、访存函数、数学函数等基础 API，以及进阶的协作组适用场景。

### 小节内容

- SIMT 编程 API 总览
- 同步与内存栅栏 API
- 原子操作 API
- Warp 函数 API
- 访存函数 API
- 数学函数 API
- 协作组 API（进阶）

### 使用前提：头文件

调用 SIMT API 前需要包含相应头文件。大部分接口包含主头文件即可，特定数据类型需额外包含对应头文件：

```cpp
#include "simt_api/asc_simt.h"   // 主头文件，覆盖大部分接口
#include "simt_api/asc_fp16.h"   // half / half2 类型
#include "simt_api/asc_bf16.h"   // bfloat16_t / bfloat16x2_t 类型
#include "simt_api/asc_fp8.h"    // fp8 相关类型
```

### SIMT 编程 API 总览

SIMT 编程 API 的接口数量很多，教学时不建议一开始逐个背诵。更好的方式是先建立一张「按用途分类」的索引地图，知道每一类 API 分别解决什么问题，遇到具体需求时再去对应类别查阅接口细节。本小节不规定固定的学习顺序，重点是帮你建立「遇到什么问题，该去哪个 API 类别找答案」的索引能力。

**SIMT 编程**场景下的功能 API 主要包含以下类别：

<table>
  <thead>
    <tr>
      <th>API 类别</th>
      <th>解决的问题</th>
      <th>代表内容或接口</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>同步与内存栅栏</td>
      <td>保证线程协作和内存可见性</td>
      <td><code>asc_syncthreads</code>、<code>asc_threadfence</code>、<code>asc_threadfence_block</code></td>
    </tr>
    <tr>
      <td>原子操作</td>
      <td>多线程安全更新共享数据</td>
      <td><code>asc_atomic_add</code>、<code>asc_atomic_cas</code>、<code>asc_atomic_max</code> 等</td>
    </tr>
    <tr>
      <td>Warp 函数</td>
      <td>Warp 内线程通信、投票和规约</td>
      <td><code>asc_all</code>、<code>asc_any</code>、<code>asc_shfl</code>、<code>asc_reduce_add</code> 等</td>
    </tr>
    <tr>
      <td>数学函数</td>
      <td>常见数学计算和类型转换</td>
      <td>float/half/bfloat16/fp8/整型数学函数</td>
    </tr>
    <tr>
      <td>访存函数</td>
      <td>控制特殊加载/存储/缓存行为</td>
      <td><code>asc_ldca</code>、<code>asc_ldcg</code>、<code>asc_stcg</code>、<code>asc_stwt</code>、<code>asc_dcci_*</code></td>
    </tr>
    <tr>
      <td>协作组（进阶）</td>
      <td>更细粒度地组织和管理协作线程组</td>
      <td><code>thread_block</code>、<code>thread_block_tile</code>、<code>coalesced_group</code></td>
    </tr>
  </tbody>
</table>

### 同步与内存栅栏 API

同步与内存栅栏 API 对应《同步机制》小节，是多线程协作正确性的基础。

<table>
  <thead>
    <tr>
      <th>子类</th>
      <th>代表接口</th>
      <th>典型用途</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>线程同步</td>
      <td><code>asc_syncthreads</code></td>
      <td>等待当前 Thread Block 内所有线程到达同步点；共享内存分阶段读写时让线程都到齐</td>
    </tr>
    <tr>
      <td>内存栅栏</td>
      <td><code>asc_threadfence</code></td>
      <td>约束全局/共享内存写入的可见顺序，保证不同线程访问的时序性</td>
    </tr>
    <tr>
      <td>块内内存栅栏</td>
      <td><code>asc_threadfence_block</code></td>
      <td>保证调用前的内存操作对同一 Thread Block 内其它线程可见</td>
    </tr>
  </tbody>
</table>

使用原则：先判断自己要解决的是「线程是否都到达同步点」，还是「写入是否按预期可见」。前者看 `asc_syncthreads`，后者看 `threadfence` 系列。

### 原子操作 API

原子操作用于解决多个线程同时更新同一地址时的数据竞争问题。这些接口可作用于 Unified Buffer 或 Global Memory 上的数据。

<table>
  <thead>
    <tr>
      <th>子类</th>
      <th>代表接口</th>
      <th>典型用途</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>加减类</td>
      <td><code>asc_atomic_add</code>、<code>asc_atomic_sub</code></td>
      <td>多线程计数</td>
    </tr>
    <tr>
      <td>赋值/比较类</td>
      <td><code>asc_atomic_exch</code>、<code>asc_atomic_cas</code></td>
      <td>安全更新共享状态、比较并交换</td>
    </tr>
    <tr>
      <td>最值类</td>
      <td><code>asc_atomic_max</code>、<code>asc_atomic_min</code></td>
      <td>多线程求最大值/最小值</td>
    </tr>
    <tr>
      <td>自增自减类</td>
      <td><code>asc_atomic_inc</code>、<code>asc_atomic_dec</code></td>
      <td>有边界条件的计数更新</td>
    </tr>
    <tr>
      <td>位运算类</td>
      <td><code>asc_atomic_and</code>、<code>asc_atomic_or</code>、<code>asc_atomic_xor</code></td>
      <td>多线程更新标志位或掩码</td>
    </tr>
  </tbody>
</table>

教学上可以把原子操作理解成「把读-改-写合成一个不可被其它线程打断的操作」。它保证正确性，但大量原子操作可能带来性能压力。

### Warp 函数 API

Warp 函数用于 Warp 内线程之间的协作。前面《线程架构》中提到，每个 Warp 包含 32 个线程；这里的 API 就是在这个粒度上进行投票、数据交换、规约和Warp内线程编号查询。

<table>
  <thead>
    <tr>
      <th>子类</th>
      <th>代表接口</th>
      <th>典型用途</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Warp Vote 类</td>
      <td><code>asc_all</code>、<code>asc_any</code>、<code>asc_ballot</code>、<code>asc_activemask</code></td>
      <td>判断 Warp 内活跃线程条件是否满足，或获取活跃线程掩码</td>
    </tr>
    <tr>
      <td>Warp Shfl 类</td>
      <td><code>asc_shfl</code>、<code>asc_shfl_up</code>、<code>asc_shfl_down</code>、<code>asc_shfl_xor</code></td>
      <td>在 Warp 内不同线程之间交换数据</td>
    </tr>
    <tr>
      <td>Warp Reduce 类</td>
      <td><code>asc_reduce_add</code>、<code>asc_reduce_max</code>、<code>asc_reduce_min</code></td>
      <td>对 Warp 内活跃线程输入做求和、最大值、最小值规约</td>
    </tr>
    <tr>
      <td>Lane-ID 类</td>
      <td><code>laneid</code>、<code>lanemask_eq</code>、<code>lanemask_lt</code>、<code>lanemask_le</code>、<code>lanemask_gt</code>、<code>lanemask_ge</code></td>
      <td>获取当前线程在 Warp 内的 lane 编号，或生成各类 lane 掩码</td>
    </tr>
  </tbody>
</table>

如果你的计算只需要 Warp 内线程协作，Warp 函数通常比把数据写到共享内存再同步更直接。

### 访存函数 API

访存函数用于控制加载、存储和缓存行为。代表接口如下：

<table>
  <thead>
    <tr>
      <th>子类</th>
      <th>代表接口</th>
      <th>典型用途</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>加载（load）</td>
      <td><code>asc_ldca</code>、<code>asc_ldcg</code></td>
      <td>按特定缓存策略从内存读取数据</td>
    </tr>
    <tr>
      <td>存储（store）</td>
      <td><code>asc_stcg</code>、<code>asc_stwt</code></td>
      <td>按特定缓存/写入策略向内存写入数据</td>
    </tr>
    <tr>
      <td>缓存控制（cache）</td>
      <td><code>asc_dcci_single</code>、<code>asc_dcci_entire</code></td>
      <td>数据缓存一致性管理</td>
    </tr>
  </tbody>
</table>

这类 API 适合在你已经理解 Global Memory、Data Cache 和 Unified Buffer 后再深入。初学阶段先知道：当普通指针访问不足以表达访存/缓存策略时，就需要查访存函数。

### 数学函数 API

数学函数目录覆盖范围很广，包含 float、half、bfloat16、fp8、整型数学库函数以及数据类型转换。

<table>
  <thead>
    <tr>
      <th>子类</th>
      <th>代表接口</th>
      <th>典型用途</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>float 类型数学库函数</td>
      <td><code>sinf</code>、<code>cosf</code>、<code>expf</code>、<code>logf</code>、<code>sqrtf</code>、<code>fmaf</code> 等</td>
      <td>单精度浮点计算</td>
    </tr>
    <tr>
      <td>half 类型</td>
      <td>half 相关算术函数</td>
      <td>半精度计算</td>
    </tr>
    <tr>
      <td>bfloat16 类型</td>
      <td>bfloat16 相关函数</td>
      <td>BF16 数据计算</td>
    </tr>
    <tr>
      <td>fp8 类型</td>
      <td>fp8 数据类型及相关支持</td>
      <td>低精度计算场景</td>
    </tr>
    <tr>
      <td>整型数学库函数</td>
      <td><code>__clz</code>、<code>__popc</code>、<code>__brev</code>、<code>max</code>、<code>min</code> 等</td>
      <td>位操作、整型辅助计算</td>
    </tr>
    <tr>
      <td>数据类型转换</td>
      <td>不同数值类型之间的转换函数</td>
      <td>精度转换或混合精度计算</td>
    </tr>
  </tbody>
</table>

教学建议：不要按目录顺序背数学函数。遇到具体计算需求时，再按数据类型和函数用途查对应条目。

### 协作组 API（进阶）

协作组（cooperative_groups）是 Ascend C SIMT 编程模型的一个**进阶扩展**，用于把协作线程组织成显式的「组」，从而控制线程协作的**粒度**。

当你需要**更细粒度的线程协作**（例如只同步块内一小撮线程、或让条件分支中到达同一路径的活跃线程协同工作）时，用协作组比自己手写内存栅栏更安全、更清晰。

典型适用场景：

- 需要在**线程块的子集**（如每 8 个、32 个线程一组）内做同步或数据交换。
- 处理**分支发散**后，让仍处于活跃路径的线程组成一个组协同计算。

协作组属于进阶特性，初学阶段了解它的用途即可。当基础 API 不足以表达复杂的分组协作逻辑时，再查阅 [SIMT 协作组 API](https://gitcode.com/cann/asc-devkit/blob/master/docs/api/SIMT-API/协作组/协作组.md) 深入学习。

## 小节小结

本小节把 **SIMT 编程**场景下的功能 API 整理成了开发者视角的学习地图：同步与内存栅栏保证多线程协作正确；原子操作解决共享数据更新；Warp 函数支持 Warp 内投票、数据交换和规约；访存函数用于表达特殊缓存/访存策略；数学函数则在具体算子开发中按需查阅；协作组作为进阶特性，提供更高层的线程分组协作能力。

到这里，SIMT 编程模型课程已经形成一条完整路径：硬件架构、线程架构、核函数、内存层级、同步机制、编程 API。后续进入真实算子开发时，可以把这份 notebook 当作概念索引和 API 导航，遇到具体需求再到对应的 API 类别中查阅接口细节。

## 课后练习

本节把 SIMT 编程 API 整理成按用途分类的学习地图，请根据学习内容完成以下题目进行自测。

1. （判断题）只要包含主头文件 `simt_api/asc_simt.h`，就能调用所有 SIMT 编程 API，无需再包含其它任何头文件。

2. （单选题）要让多个线程安全地累加同一个计数器，应优先查哪类 API？  
    A. Warp 函数  
    B. 原子操作  
    C. 同步与内存栅栏  
    D. 数学函数  

3. （单选题）`asc_shfl_down`、`asc_reduce_add`、`asc_ballot` 这些接口属于哪一类 API？  
    A. 同步与内存栅栏  
    B. 访存函数  
    C. Warp 函数  
    D. 原子操作  

4. （多选题）以下关于 SIMT 编程 API 分类的说法，哪些是正确的？  
    A. `asc_syncthreads`、`asc_threadfence` 属于同步与内存栅栏 API，用于保证线程协作与内存可见性  
    B. `thread_block_tile`、`coalesced_group` 属于协作组，用于组织协作线程组  
    C. 当普通指针访问不足以表达缓存/访存策略时，可查访存函数（如 `asc_ldcg`、`asc_stcg`）  
    D. `asc_atomic_add`、`asc_atomic_cas` 属于 Warp 函数，用于 Warp 内线程数据交换  

**执行以下代码获取答案。**

## 参考答案

运行下面的单元格查看课后练习参考答案。


In [ ]:
!cat answer/03.04.06_answer.txt
